In [ ]:
suppressPackageStartupMessages(library(rtracklayer))
suppressPackageStartupMessages(library(plyranges))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(GenomeInfoDb))
suppressPackageStartupMessages(library(BSgenome.Mfascicularis.NCBI.5.0))

载入需要的程序包：GenomicRanges

载入需要的程序包：stats4



载入需要的程序包：BiocGenerics

载入需要的程序包：generics


载入程序包：‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



载入程序包：‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


载入需要的程序包：S4Vectors


载入程序包：‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


载入需要的程序包：IRanges

载入需要的程序包：Seqinfo


载入程序包：‘plyra

In [2]:
{
# Prepare Genome ----------------------------------------------------------1
suppressPackageStartupMessages(library(BSgenome.Mfascicularis.NCBI.5.0))
assembly_data <- "GCA_000364345.1_Macaca_fascicularis_5.0_assembly_report.tsv"
chrominfo <- if(file.exists(assembly_data)) {
    read.delim(assembly_data)
}else{ 
    chrominfo_data <- getChromInfoFromNCBI("Macaca_fascicularis_5.0")
    chrominfo_data %>% write.table(assembly_data, sep = '\t', row.names = F, col.names = T, quote = F)
    chrominfo_data
}
seqlevels(BSgenome.Mfascicularis.NCBI.5.0) <- setNames(chrominfo$UCSCStyleName, chrominfo$SequenceName)

seqinfo_macFas5 <- seqinfo(BSgenome.Mfascicularis.NCBI.5.0) %>% as.data.frame %>% 
    tibble::rownames_to_column("seqnames") %>% 
    filter(grepl('^chr', seqnames)) %>% tibble %>% 
    {Seqinfo(seqnames = .$seqnames, 
                seqlengths = .$seqlengths, 
                isCircular = .$isCircular, 
                genome = 'macFas5')}

# Prepare Genome Annotation -----------------------------------------------2
genom.blacklist <- readRDS('/mnt/112-rawdata-112/macaque/Macaca_fascicularis_5.0.91.2.2_new/macFas5.huada.backlist.Rds') 
genome.annotation <- rtracklayer::import('/mnt/112-rawdata-112/macaque/Macaca_fascicularis_5.0.91.2.2_new/Macaca_fascicularis_5.0.91.2.2_withchrname.gtf') %>% 
    plyranges::mutate(tx_id = transcript_id, tx_name = transcript_name) %>% 
    plyranges::mutate(gene_name = if_else(is.na(gene_name),gene_id,gene_name)) %>% 
    plyranges::filter(! seqnames %in% c("M","MT","Y","chrY","chrMT","chrM"))

### filter annotations for genes mapped to multiple chromosomes
{
    exclude_genename <- genome.annotation %>% data.frame() %>% group_by(gene_name) %>%
        summarise(N_chr = n_distinct(seqnames), .groups = "drop") %>%
        dplyr::filter(N_chr > 1) %>% pull(gene_name)
    genome.annotation <- genome.annotation %>% plyranges::filter(!gene_name %in% exclude_genename)
    seqlevels(genome.annotation) <- GenomeInfoDb::seqlevelsInUse(genome.annotation)
    rm(exclude_genename)
}
    
} %>% invisible %>% suppressMessages %>% suppressWarnings
### Save Image 
# save.image('macFas5_genomeInfo_signac-new.RData')

In [7]:
save.image('macFas5_genomeInfo_signac-new.RData')